# Treinamento completo da U-Net no Colab (GPU T4)

6 classes, validação multi-tile, amostragem de classes raras, focal loss, augmentation, early stopping e checkpoint no Drive.

## Antes de rodar
1. No Drive, crie `MyDrive/georural/` e envie `georural.zip` (repositório) e `unet_dataset_full.zip` (tiles).
2. Ative a GPU: *Runtime > Change runtime type > T4 GPU*.
3. Rode as células em ordem.

O melhor modelo fica em `MyDrive/georural/brazil_unet.pth` e o checkpoint em `brazil_unet_last.pth`.

In [ ]:
import glob
import os
import shutil
import sys
import zipfile

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/georural'
os.makedirs(DRIVE_DIR, exist_ok=True)
REPO_ZIP = f'{DRIVE_DIR}/georural.zip'
DATA_ZIP = f'{DRIVE_DIR}/unet_dataset_full.zip'

if os.path.exists(REPO_ZIP):
    if os.path.isdir('GeoRural'):
        shutil.rmtree('GeoRural')
    with zipfile.ZipFile(REPO_ZIP) as z:
        z.extractall('.')

if os.path.isdir('GeoRural'):
    os.chdir('GeoRural')

unet_dir = 'data/processed/unet_full'
if os.path.isdir(unet_dir):
    shutil.rmtree(unet_dir)
os.makedirs(unet_dir, exist_ok=True)
if os.path.exists(DATA_ZIP):
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall(unet_dir)

sys.path.insert(0, os.getcwd())
print('Projeto em:', os.getcwd())
print('Tiles:', len(glob.glob('data/processed/unet_full/*_image.npy')))

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('AVISO: GPU não ativa. Ative Runtime > Change runtime type > T4 GPU.')

In [ ]:
import numpy as np
from src.classifiers.unet_cnn import (
    build_unet, load_unet_tiles, make_patch_datasets, train_unet,
)
from src.config import IGNORE_LABEL, NUM_CLASSES

tiles = load_unet_tiles('data/processed/unet_full')

# Tiles excluídos do treino:
#  - unet_solo_lencois: dunas + lagoas (areia nua NDVI~0.01 vs água rasa
#    NDVI~0.13) são quase idênticas e ensinam 'superfície escura = água'.
EXCLUDE_TILES = {'unet_solo_lencois'}
tiles = {k: v for k, v in tiles.items() if k not in EXCLUDE_TILES}

print('Total de tiles:', len(tiles))
print('Excluídos:', sorted(EXCLUDE_TILES))
print('Tiles:', list(tiles))

import json
with open('data/processed/unet_full/dataset_report.json', encoding='utf-8') as f:
    dataset_report = json.load(f)
print('Tiles válidos no relatório:', len(dataset_report['valid_tiles']))
ndvi_min = min(float(np.nanmin(img[..., 6])) for img, _ in tiles.values())
ndvi_max = max(float(np.nanmax(img[..., 6])) for img, _ in tiles.values())
classes = set()
for _, labels in tiles.values():
    classes.update(int(value) for value in np.unique(labels))
print('NDVI global:', ndvi_min, ndvi_max)
print('Classes presentes:', sorted(classes))

# paracatu fica no TREINO (mineração de ouro escura NDVI~0.24) para o modelo
# aprender a separar mineração de água; quadrilatero testa essa generalização.
VAL_TILES = [
    tile for tile in (
        'unet_piracicaba_t0',
        'unet_caatinga_t2',
        'unet_rio_verde_t2',
        'unet_cerrado_t0',
        'unet_solo_quadrilatero',
    ) if tile in tiles
]
if not VAL_TILES:
    VAL_TILES = [next(iter(tiles))]
print('Validação multi-tile:', VAL_TILES)
for name in VAL_TILES:
    values, counts = np.unique(tiles[name][1], return_counts=True)
    dist = {int(v): int(c) for v, c in zip(values, counts)}
    print(f'  {name}: {dist}')

train_ds, val_ds = make_patch_datasets(
    tiles, VAL_TILES, patch_size=64, stride=32, include_indices=True,
    rare_classes=(3, 4, 5), rare_patches_per_class=100,
    rare_min_fraction=0.01, seed=42,
)
print('Treino:', len(train_ds), 'patches | Validação:', len(val_ds), 'patches')
class_weights = train_ds.class_weights(exponent=0.5)
print('Pesos (1/sqrt(freq), média 1):', np.round(class_weights, 4).tolist())

# Sanidade: confirma que patches 100% vazios (255) não serão amostrados.
sw = train_ds.sample_weights(IGNORE_LABEL)
print('Patches vazios (peso 0):', int((sw == 0).sum()), '/', len(sw))
assert int((sw == 0).sum()) < len(sw), 'Nenhum patch válido para treinar!'

In [ ]:
import json

SAVE_PATH = f'{DRIVE_DIR}/brazil_unet.pth'
CHECKPOINT = f'{DRIVE_DIR}/brazil_unet_last.pth'

model = build_unet(
    input_shape=(64, 64, 15), num_classes=NUM_CLASSES
)
history = train_unet(
    model, train_ds, None, val_ds, None,
    epochs=100, batch_size=16, learning_rate=0.001,
    save_path=SAVE_PATH,
    class_weights=class_weights,
    ignore_index=IGNORE_LABEL,
    augment=True,
    balance=True,
    patience=20,
    focal_gamma=1.5,
    checkpoint_path=CHECKPOINT,
    resume=False,  # mude para True só para retomar uma sessão interrompida
    checkpoint_every=5,
)

os.makedirs('outputs/full', exist_ok=True)
with open('outputs/full/unet_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(history.get('metrics', {}), f, indent=2, ensure_ascii=False)

with open(f'{DRIVE_DIR}/unet_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(history.get('metrics', {}), f, indent=2, ensure_ascii=False)

print(json.dumps(history.get('metrics', {}), indent=2, ensure_ascii=False))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train')
plt.plot(history['val_loss'], label='Val')
plt.title('Loss')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train')
plt.plot(history['val_acc'], label='Val')
plt.title('Accuracy')
plt.legend()
plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/unet_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

import shutil
from google.colab import files
local_model = '/content/brazil_unet.pth'
shutil.copy(SAVE_PATH, local_model)
files.download(local_model)